# 12 — Employee Skills Table

**Day 3, Step 12.** The Build Notes anticipated this exact problem:

> *"My current five datasets might not actually contain what skills each
> employee currently has... If it doesn't → build a controlled table for the MVP
> so the rest of the pipeline has something real to work with."*

Confirmed — **finding F3**: none of the five files record individual skills.

The table is built to three rules so it is defensible rather than decorative:
**derived not random**, **deterministic**, and **labelled everywhere**.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.skills.employee_skills import build_employee_skills

skills = build_employee_skills()
print(f"{len(skills):,} rows  |  {skills['employee_id'].nunique():,} employees")
skills.groupby(["population", "tier"]).agg(
    rows=("skill_name", "size"), hold_rate=("holds_skill", "mean")).round(3)

2026-08-28 01:55:07 | INFO  | crosswalk resolved


2026-08-28 01:55:07 | INFO  | role requirements built


2026-08-28 01:55:07 | INFO  | employee skills derived


156,314 rows  |  3,717 employees


rows  hold_rate
population tier                          
A          foundational  14700      1.000
           technical     36750      0.455
B          foundational  29864      1.000
           technical     75000      0.373

## Deterministic, not random

Individual variation comes from a hash of `(seed, employee_id, skill)`, not from
a random number generator. No global state, no dependence on row order,
reproducible across processes and machines.

In [3]:
print("rebuild is byte-identical:", skills.equals(build_employee_skills()))
print("every row labelled derived:", bool(skills["is_derived"].all()))

2026-08-28 01:55:07 | INFO  | crosswalk resolved


2026-08-28 01:55:07 | INFO  | role requirements built


2026-08-28 01:55:07 | INFO  | employee skills derived


rebuild is byte-identical: True
every row labelled derived: True


## `person_key`, not `employee_id`

Population A spans IDs 1–2068 and Population B spans 1001–4000, so **753 numeric
IDs exist in both — for entirely different people**. A `groupby("employee_id")`
would silently merge them: finding F1 in a new guise. `person_key` encodes the
population and is the only safe identity downstream.

In [4]:
print(skills[["person_key", "population", "employee_id", "skill_name",
              "tier", "proficiency_level", "holds_skill", "is_derived"]].head(8))
print("\nperson_key never spans populations:",
      bool((skills.groupby('person_key')['population'].nunique() == 1).all()))

  person_key population  employee_id             skill_name          tier  proficiency_level  holds_skill  is_derived
0        A-1          A            1        Active Learning  foundational               3.15            1        True
1        A-1          A            1       Active Listening  foundational               4.31            1        True
2        A-1          A            1      Critical Thinking  foundational                3.0            1        True
3        A-1          A            1    Learning Strategies  foundational               2.77            1        True
4        A-1          A            1            Mathematics  foundational               2.69            1        True
5        A-1          A            1             Monitoring  foundational               3.07            1        True
6        A-1          A            1  Reading Comprehension  foundational               3.42            1        True
7        A-1          A            1                Scie